In [4]:
!pip install gradio

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

df = pd.read_csv("Updated_sales.csv")

# Clean Data
df = df.dropna(how='all')
df = df[df['Order Date'] != 'Order Date']

# Build User-Item Profile
user_item_matrix = df.groupby('Purchase Address')['Product'].apply(lambda x: ', '.join(x)).reset_index()
user_item_matrix['product_descriptions'] = user_item_matrix['Product']

# Compute NLP Matrices (Runs ONCE at startup)
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(user_item_matrix['product_descriptions'])
similarity_matrix = cosine_similarity(tfidf_matrix)

# Optimize the Dropdown: Only show customers who bought > 1 item so we have good test cases
purchase_counts = df['Purchase Address'].value_counts()
valid_customers = purchase_counts[purchase_counts > 1].index.tolist()
dropdown_choices = valid_customers[:1000] # Cap at 1000 to keep the browser UI fast

print("Engine Ready!")

def get_ui_recommendations(target_address, num_recommendations):
    if target_address not in user_item_matrix['Purchase Address'].values:
        return "Address not found.", "No recommendations available."

    target_customer_index = user_item_matrix[user_item_matrix['Purchase Address'] == target_address].index[0]

    # Fetch past purchases to display on the UI
    past_purchases_str = user_item_matrix.loc[target_customer_index, 'Product']
    target_customer_purchases = set(past_purchases_str.split(', '))

    # Find similar users
    similar_customers = similarity_matrix[target_customer_index].argsort()[::-1][1:(num_recommendations * 3) + 1]

    recommendations = []
    for customer_index in similar_customers:
        customer_purchases = set(user_item_matrix.iloc[customer_index]['Product'].split(', '))
        new_items = customer_purchases.difference(target_customer_purchases)
        recommendations.extend(new_items)

        # Stop searching if we hit the requested number
        if len(set(recommendations)) >= num_recommendations:
            break

    recs = list(set(recommendations))[:num_recommendations]

    # Formatting for the Gradio Textbox
    formatted_past = "\n".join([f"✔️ {item}" for item in target_customer_purchases])
    if not recs:
        formatted_recs = "User has already purchased everything similar users bought!"
    else:
        formatted_recs = "\n".join([f"🚀 {item}" for item in recs])

    return formatted_past, formatted_recs

# Using Blocks allows for custom layouts instead of just stacked inputs/outputs
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛒 E-Commerce Recommender System")
    gr.Markdown("Select a target customer to analyze their purchase history and generate collaborative-filtering recommendations based on lookalike audiences.")

    with gr.Row():
        # Left Column (Controls)
        with gr.Column(scale=1):
            user_dropdown = gr.Dropdown(
                choices=dropdown_choices,
                label="Target Customer Address",
                value=dropdown_choices[0] if dropdown_choices else None,
                info="Sample of high-frequency customers"
            )
            num_recs = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Number of Recommendations")
            submit_btn = gr.Button("Generate Recommendations", variant="primary")

        # Right Column (Results)
        with gr.Column(scale=2):
            past_purchases_out = gr.Textbox(label="Customer's Past Purchases", lines=4, interactive=False)
            recommendations_out = gr.Textbox(label="AI Recommended Products", lines=5, interactive=False)

    # Wire the button to the function
    submit_btn.click(
        fn=get_ui_recommendations,
        inputs=[user_dropdown, num_recs],
        outputs=[past_purchases_out, recommendations_out]
    )

# Launch the app with a public share link so you can view it outside of Colab
demo.launch(share=True)

Loading data and building similarity matrices. Please wait...
Engine Ready!


/tmp/ipykernel_1927/3161024287.py:73: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fe7e1480bd5327272.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
